In [ ]:
import arcpy
import os

# -------------------------------------------------------
# INPUT PATHS — update these
# -------------------------------------------------------
input_gdb  = r"C:\path\to\your\input.gdb"
output_gdb = r"C:\path\to\your\output.gdb"

road_fc = os.path.join(input_gdb, "Road_QA")

# -------------------------------------------------------
# ROAD FIELD NAMES — update to match your data
# -------------------------------------------------------
ROAD_FL_FIELD = "F_Addr_L_911"
ROAD_TL_FIELD = "T_Addr_L_911"
ROAD_FR_FIELD = "F_Addr_R_911"
ROAD_TR_FIELD = "T_Addr_R_911"

# -------------------------------------------------------
# OUTPUT
# -------------------------------------------------------
road_result = os.path.join(output_gdb, "Road_Parity_Result")

print("Configuration loaded.")

In [ ]:
print("Copying road layer to output GDB...")

if arcpy.Exists(road_result):
    arcpy.Delete_management(road_result)

arcpy.CopyFeatures_management(road_fc, road_result)

for fname, flength in [
    ("Parity_L",    20),
    ("Parity_R",    20),
    ("Parity_Diff", 10),
]:
    arcpy.AddField_management(road_result, fname, "TEXT", field_length=flength)

print("Road layer copied and parity fields added.")

In [ ]:
def get_side_parity(f_val, t_val):
    """
    Derive parity for one side of a road segment.
    Returns Even, Odd, Both, or Undetermined.
    """
    values = []
    for v in [f_val, t_val]:
        if v is not None:
            try:
                values.append(int(float(v)))
            except:
                pass

    if not values:
        return "Undetermined"

    parities = set("Even" if v % 2 == 0 else "Odd" for v in values)

    if parities == {"Even"}:
        return "Even"
    elif parities == {"Odd"}:
        return "Odd"
    else:
        return "Both"


def get_diff(parity_l, parity_r):
    """
    Compare left and right side parity.
    Yes  = both sides same parity (suspicious)
    No   = sides differ (correct alternating pattern)
    Unknown = one or both sides undetermined
    """
    if parity_l == "Undetermined" or parity_r == "Undetermined":
        return "Unknown"
    return "Yes" if parity_l == parity_r else "No"


print("Helper functions defined.")

In [ ]:
print("Running parity check...")

road_fields = [
    ROAD_FL_FIELD, ROAD_TL_FIELD,
    ROAD_FR_FIELD, ROAD_TR_FIELD,
    "Parity_L", "Parity_R", "Parity_Diff"
]

with arcpy.da.UpdateCursor(road_result, road_fields) as cur:
    for row in cur:
        try:
            fL = float(row[0]) if row[0] is not None else None
        except:
            fL = None
        try:
            tL = float(row[1]) if row[1] is not None else None
        except:
            tL = None
        try:
            fR = float(row[2]) if row[2] is not None else None
        except:
            fR = None
        try:
            tR = float(row[3]) if row[3] is not None else None
        except:
            tR = None

        parity_l = get_side_parity(fL, tL)
        parity_r = get_side_parity(fR, tR)
        diff     = get_diff(parity_l, parity_r)

        row[4] = parity_l
        row[5] = parity_r
        row[6] = diff

        cur.updateRow(row)

print("Parity check complete.")

In [ ]:
# Count results
total    = int(arcpy.GetCount_management(road_result)[0])
diff_yes = 0
diff_no  = 0
diff_unk = 0

with arcpy.da.SearchCursor(road_result, ["Parity_Diff"]) as cur:
    for row in cur:
        if row[0] == "Yes":
            diff_yes += 1
        elif row[0] == "No":
            diff_no += 1
        else:
            diff_unk += 1

print("=" * 50)
print("Parity Check Complete")
print("=" * 50)
print(f"  Total road segments     : {total}")
print(f"  Parity agrees (No diff) : {diff_no}")
print(f"  Parity differs (Yes)    : {diff_yes}")
print(f"  Unknown                 : {diff_unk}")
print("-" * 50)
print(f"  Road_Parity_Result → {road_result}")
print("=" * 50)